# Stage3 full v1 training: first epoch

156 training videos, 31 validation videos; existing route split. Pinned submission requirements, T4, batch2, stride8, learning rate1e-4, random initialization. Save checkpoint and dense validation predictions.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, tempfile, zipfile

INPUT = Path('/kaggle/input')
WORK_PARENT = Path('/kaggle/working')
# Set SOURCE explicitly only when several matching bundles are attached.
SOURCE = None
if SOURCE is None:
    archives = list(INPUT.rglob('stage3-training-v1.zip'))
    directories = [p.parent for p in INPUT.rglob('bundle.json')
                   if (p.parent/'src/stage3_pipeline.py').is_file()
                   and (p.parent/'dataset/split_manifest.csv').is_file()]
    candidates = archives + directories
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Stage3 bundle; set SOURCE to one of: {candidates}')
    SOURCE = candidates[0]
SOURCE = Path(SOURCE)
WORK = Path(tempfile.mkdtemp(prefix='stage3-gpu-', dir=WORK_PARENT))
if SOURCE.is_file():
    with zipfile.ZipFile(SOURCE) as z:
        for member in z.infolist():
            rel = PurePosixPath(member.filename)
            if rel.is_absolute() or '..' in rel.parts or '\\' in member.filename:
                raise ValueError('Unsafe ZIP member')
            if not (WORK/member.filename).resolve().is_relative_to(WORK.resolve()):
                raise ValueError('ZIP path escapes workspace')
        z.extractall(WORK)
else:
    shutil.copytree(SOURCE, WORK, dirs_exist_ok=True)
BUNDLE = json.loads((WORK/'bundle.json').read_text())
if BUNDLE.get('kind') != 'STAGE3_FULL_V1':
    raise ValueError('Not a Stage3 smoke bundle')
for relative, expected in BUNDLE['file_sha256'].items():
    path = (WORK/relative).resolve()
    if not path.is_relative_to(WORK.resolve()):
        raise ValueError('Manifest path escapes workspace')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Bundle checksum mismatch: {relative}')
print('Verified bundle:', SOURCE)
print('Working directory:', WORK)


In [ ]:
import subprocess, sys, time, os
VENV = Path(tempfile.mkdtemp(prefix='stage3-pinned-', dir='/tmp'))
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
started = time.monotonic()
with (WORK/'install.log').open('w') as log:
    installed = subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--no-cache-dir', '-r', str(WORK/'requirements.txt')], stdout=log, stderr=subprocess.STDOUT)
INSTALL = {'exit_code': installed.returncode, 'seconds': time.monotonic()-started,
           'requirements_sha256': hashlib.sha256((WORK/'requirements.txt').read_bytes()).hexdigest()}
(WORK/'install.json').write_text(json.dumps(INSTALL, indent=2))
if installed.returncode:
    print((WORK/'install.log').read_text()[-12000:])
    raise RuntimeError('Pinned requirements installation failed')
print('Installation:', INSTALL)
RUNNER = "from pathlib import Path\nimport sys,json,time\nimport pandas as pd\nimport torch\nimport importlib.metadata\nWORK=Path(sys.argv[1]);sys.path.insert(0,str(WORK/'src'))\nimport stage3_pipeline as pipeline\nversions={}\nfor line in (WORK/'requirements.txt').read_text().splitlines():\n    if not line.strip() or line.startswith('#'):continue\n    name,expected=line.split('==');actual=importlib.metadata.version(name)\n    assert actual.split('+')[0]==expected\n    versions[name]=actual\nfull=pipeline.load_records(WORK/'dataset')\nsubset=WORK/'dataset'\nselected=full\nassert torch.cuda.is_available()\ntorch.cuda.reset_peak_memory_stats();started=time.monotonic()\nsys.argv=['stage3_pipeline.py','train','--dataset-dir',str(subset),'--output-dir',str(WORK/'training-run'),'--device','cuda','--epochs','1','--batch-size','2','--train-stride','8','--from-scratch']\npipeline.main()\nreport=dict(status='PASS',scope='Full calibrated v1 dataset; first epoch from scratch, stride8 batch2',seconds=time.monotonic()-started,peak_allocated_bytes=torch.cuda.max_memory_allocated(),peak_reserved_bytes=torch.cuda.max_memory_reserved(),gpu=torch.cuda.get_device_name(0),versions=versions,full_audit={k:dict(videos=len(v),samples=sum(len(r['accel']) for r in v)) for k,v in full.items()},selected_ids={k:[r['ID'] for r in v] for k,v in selected.items()})\n(WORK/'training-report.json').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n"
(WORK/'training_runner.py').write_text(RUNNER)
with (WORK/'training.log').open('w') as log:
    process=subprocess.Popen([PYTHON,'-u',str(WORK/'training_runner.py'),str(WORK)],cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    returncode=process.wait()
if returncode:raise RuntimeError('Training failed; retain training.log')
with zipfile.ZipFile(WORK/'stage3-training-result.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['training-report.json','training.log','training_runner.py','bundle.json','install.json','training-run/history.json','training-run/evaluation.json','training-run/data_fingerprints.json','training-run/validation_predictions.csv']:
        z.write(WORK/name,name)
print('Full first epoch complete; checkpoint:',WORK/'training-run/model/stage3/best.pt')
